---
title: "Chapter 14: Validation and Limitations"
short_title: "14. Validation and limitations"
---

## 1. Automatic checks (`validate_m2.py`)

After parsing, `validate_m2.py` re-checks every result of each range, without sending any request. Each check yields
`PASS`, `WARN` (worth reviewing, possibly caused by the paper's source) or `FAIL` (wrong data). The main groups of
checks:

| Group | Checks |
| :--- | :--- |
| Files and headers | every file present in `parsed/`, correct columns, every Milestone 1 version has a record |
| Tree | exactly one root per version, the root is never a child, every node reaches the root, no cycle, **exactly one parent per node**, no duplicate edge, leaves have no children, no orphan element |
| Content | valid node kinds, no bibliography inside the tree, no broken characters |
| References | every `\cite` key exists in the `bibtex` table (`citation_key_coverage`) |
| Parquet tables | correct schema, no nulls in required columns, row counts equal to the JSON files |

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils_m2 import (COLORS, KIND_LABELS, PROBLEM_LABELS, group, human_bytes, human_seconds, load,
                             members, meta, metric, metric_table, num, ordered, pct, percent, ready, secs,
                             show, table)
READY = ready()

In [ ]:
if READY:
    vc = load("validation_counts")
    st = vc.pivot_table(index="member", columns="status", values="n", aggfunc="sum", fill_value=0)
    for s in ["PASS", "WARN", "FAIL"]:
        if s not in st.columns:
            st[s] = 0
    st = ordered(st[["PASS", "WARN", "FAIL"]].reset_index())
    total = st[["PASS", "WARN", "FAIL"]].sum()
    st = pd.concat([st, pd.DataFrame([{"member": "Group", **total.to_dict()}])], ignore_index=True)
    for s in ["PASS", "WARN", "FAIL"]:
        st[s] = st[s].map(num)
    st.columns.name = None
    table(st.rename(columns={"member": "Range"}), "Table 14.1. Check results per range (number of checks)")
    bad = vc[vc["status"] != "PASS"]
    if bad.empty:
        print("No check at WARN or FAIL level.")
    else:
        by = bad.groupby(["check_name", "status"])["n"].sum().reset_index().sort_values("n", ascending=False)
        fig = px.bar(by, x="n", y="check_name", color="status", orientation="h", color_discrete_sequence=COLORS,
                     title="Checks with WARN or FAIL, whole group", labels={"n": "Checks", "check_name": ""})
        fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=max(300, 40 * by["check_name"].nunique()))
        show(fig)

The numbers are **checks**, not papers: one paper can produce several checks, one per version.

## 2. FAIL cases

In [ ]:
if READY:
    iss = load("validation_issues")
    fail = iss[iss["status"] == "FAIL"]
    if fail.empty:
        print("No check failed.")
    else:
        table(ordered(fail)[["member", "arxiv_id", "version", "check_name", "details"]].fillna("–").rename(
            columns={"member": "Range", "arxiv_id": "Paper", "version": "Version", "check_name": "Check", "details": "Details"}),
            f"Table 14.2. {len(fail)} failed checks")

**Paper 2605.16616 (Quan's range).** After the official run, the group reviewed by hand the only paper reported as
failing:

- **The FAIL of `bibliography_exclusion` is a false positive.** This check searches the tree's content for
  `\begin{thebibliography}` to make sure the bibliography did not leak into the tree. The paper has a figure that
  prints a `.bbl` file verbatim as an example, so the string lies in the content of a figure in the body. The parser
  is right to keep that figure; the data is correct.
- **One extra reference.** An appendix of the paper (page 21) prints the `.bbl` file of another paper verbatim; the
  parser read one entry of it (`aitchison1975statistical`) as a reference of this paper. The paper has 48 entries:
  47 are correct, 1 is extra. Milestone 3 drops the pair (`2605.16616`, `aitchison1975statistical`) when loading the
  `bibtex` table; the Milestone 2 data is left unchanged.

Both cases come from a paper reprinting the LaTeX code of a bibliography. The check reports no other paper, so the
group **did not change the parser or re-run it** and records them here.

## 3. Common warnings

In [ ]:
if READY:
    warn = iss[iss["status"] == "WARN"]
    if warn.empty:
        print("No warning.")
    else:
        pv = warn.groupby(["check_name", "member"]).size().reset_index(name="n")
        pv = pv.pivot_table(index="check_name", columns="member", values="n", fill_value=0)
        pv = pv[[m for m in members()["member"] if m in pv.columns]]
        pv["Group"] = pv.sum(axis=1)
        pv.columns.name = None
        pv = pv.astype(int).sort_values("Group", ascending=False).reset_index().rename(columns={"check_name": "Check"})
        table(pv, "Table 14.3. Warnings per check and range")

- `gathering_problems`: a version had a problem while merging files (several root candidates, a guessed root file…).
  This count includes the versions without TeX, whose Milestone 1 status is recorded as a problem. For the others,
  the tree is still built from the text that could be read ([Chapter 9](./09_parsing.ipynb)).
- `citation_key_coverage`: a `\cite` points to a key that is not in the bibliography, usually because the authors
  cite an undefined key, or the bibliography is in a `.bib`/`.bbl` that was not submitted.
- `empty_tex_tree`: a version with TeX but an empty tree, usually because an included file is missing from the source.
- `nonstandard_leaf_kinds`: a heading, list item or `\paragraph` node without children, so that it is a leaf of the
  tree although the leaves should be sentences, formulas or figures.
- `appendix_included`, `acknowledgements_included`: the source contains an `\appendix` command or an acknowledgements
  section, but no heading with that word was found in the tree of the version. The sources behind the 99 warnings
  were inspected: in 65 the command stands after `\end{document}`, where LaTeX does not read it, and in 26 it first
  occurs in the preamble, for example inside a macro definition.
- `encoding_replacement_chars`: the parsed content holds the Unicode replacement character.

A check that passes does not guarantee that every citation is matched correctly: the `WARN` cases were sampled and
reviewed with `audit_m2_warnings.py` and `diagnose_m2_warnings.py` to separate problems of the source from problems
of the parser.

## 4. Papers that could not be parsed

In [ ]:
if READY:
    p = load("papers")
    failed = p[p["status"] != "done"]
    if failed.empty:
        print(f"Every paper ({len(p):,}) was parsed (status done).")
    else:
        t = failed.groupby(["member", "error_type"], dropna=False).size().reset_index(name="Papers")
        table(t.rename(columns={"member": "Range", "error_type": "Error type"}), "Table 14.4. Papers that could not be parsed")

## 5. Limitations

- **Missing source files:** when a paper calls a `.tex`, `.bib` or `.bbl` file that is not in its arXiv source
  package, that part cannot be read; the group records it instead of guessing the content.
- **Free-form references:** `\bibitem` has no fixed structure, so titles, authors and years are extracted by rules
  and can be wrong for unusual formats, for example a year taken from a page range. A title of fewer than 3 words is
  too ambiguous to merge entries by title, so such an entry can appear twice when it shares no key, DOI, arXiv ID or
  URL.
- **References left in the tree:** `hierarchy.py` does not recognize `mcitethebibliography`, `CSLReferences` or
  `\thebibliography` used as a command. Parsing the papers that use these forms again shows reference entries in the
  tree of at least 16 papers (13, 1 and 2 respectively), which the `bibliography_exclusion` check does not detect. A
  list typed by hand under a heading that is not a section command can remain in the tree as well. Milestone 3
  matches references from the `bibtex` table only, not from the tree, so the group records this limitation instead
  of fixing it and re-running everything.
- **References typed by hand:** a scan of the sources found 81 papers that type their reference list without
  `thebibliography`, `.bib` or amsrefs. No BibTeX entry is extracted from them.
- **Exact deduplication:** two sentences that differ by one punctuation mark or one capital letter are different, as
  required by the statement ("identical content"); the group does not use approximate matching for content.
- **Performance measurements:** RAM is a sampled value; times depend on the Colab machine assigned to each run.